In [0]:
%skip
df = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("/Volumes/uber_pyspark_dbt/source/raw_source_data/vehicles/")


In [0]:
%skip
schema_customer = df.schema
schema_customer

## Entities and Schema Array

In [0]:
entity_list = [
    {"entity_name":"customers",
     "schema":"StructType([StructField('customer_id', IntegerType(), True), StructField('first_name', StringType(), True), StructField('last_name', StringType(), True), StructField('email', StringType(), True), StructField('phone_number', StringType(), True), StructField('city', StringType(), True), StructField('signup_date', DateType(), True), StructField('last_updated_timestamp', TimestampType(), True)])"},
    {"entity_name":"drivers",
     "schema":"StructType([StructField('driver_id', IntegerType(), True), StructField('first_name', StringType(), True), StructField('last_name', StringType(), True), StructField('phone_number', StringType(), True), StructField('vehicle_id', IntegerType(), True), StructField('driver_rating', DoubleType(), True), StructField('city', StringType(), True), StructField('last_updated_timestamp', TimestampType(), True)])"},
    {"entity_name":"locations",
     "schema":"StructType([StructField('location_id', IntegerType(), True), StructField('city', StringType(), True), StructField('state', StringType(), True), StructField('country', StringType(), True), StructField('latitude', DoubleType(), True), StructField('longitude', DoubleType(), True), StructField('last_updated_timestamp', TimestampType(), True)])"},
    {"entity_name":"payments",
     "schema":"StructType([StructField('payment_id', IntegerType(), True), StructField('trip_id', IntegerType(), True), StructField('customer_id', IntegerType(), True), StructField('payment_method', StringType(), True), StructField('payment_status', StringType(), True), StructField('amount', DoubleType(), True), StructField('transaction_time', TimestampType(), True), StructField('last_updated_timestamp', TimestampType(), True)])"},
    {"entity_name":"trips",
     "schema":"StructType([StructField('trip_id', IntegerType(), True), StructField('driver_id', IntegerType(), True), StructField('customer_id', IntegerType(), True), StructField('vehicle_id', IntegerType(), True), StructField('trip_start_time', TimestampType(), True), StructField('trip_end_time', TimestampType(), True), StructField('start_location', StringType(), True), StructField('end_location', StringType(), True), StructField('distance_km', DoubleType(), True), StructField('fare_amount', DoubleType(), True), StructField('payment_method', StringType(), True), StructField('trip_status', StringType(), True), StructField('last_updated_timestamp', TimestampType(), True)])"},
    {"entity_name":"vehicles",
     "schema":"StructType([StructField('vehicle_id', IntegerType(), True), StructField('license_plate', StringType(), True), StructField('model', StringType(), True), StructField('make', StringType(), True), StructField('year', IntegerType(), True), StructField('vehicle_type', StringType(), True), StructField('last_updated_timestamp', TimestampType(), True)])"}
]

## Dynamic read and write streams

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType, DoubleType

base_volume_loc = "/Volumes/uber_pyspark_dbt/source/raw_source_data"
bronze_chkpoint_vol = "/Volumes/uber_pyspark_dbt/bronze/_checkpoints"
try:
    for entity in entity_list:
        entity_name = entity["entity_name"]
        schema = eval(entity["schema"])

        df = spark.readStream.format("csv")\
            .option("header", True)\
            .schema(schema)\
            .load(f"{base_volume_loc}/{entity_name}/")

        df.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation",f"{bronze_chkpoint_vol}/{entity_name}")\
            .trigger(once=True)\
            .toTable(f"uber_pyspark_dbt.bronze.{entity_name}")
except Exception as e:
    print(e)

In [0]:
%skip
df = spark.readStream.format("csv")\
    .option("header", True)\
    .schema(schema_customer)\
    .load("/Volumes/uber_pyspark_dbt/source/raw_source_data/customers/")

In [0]:
%skip
df.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","/Volumes/uber_pyspark_dbt/source/raw_source_data/customers/_checkpoints")\
    .trigger(once=True)\
    .toTable("uber_pyspark_dbt.bronze.customers")